In [ ]:
import cv2
import numpy as np
import random
from imutils import face_utils
from FaceDetectors import FaceDetector

In [ ]:
def get_mouth_aspect_ratio(mouth_points):
    """Calcula el ratio de aspecto de la boca (MAR) a partir de los puntos internos."""
    A = np.linalg.norm(mouth_points[2] - mouth_points[6]) # Distancia vertical interna 1
    B = np.linalg.norm(mouth_points[3] - mouth_points[5]) # Distancia vertical interna 2
    C = np.linalg.norm(mouth_points[0] - mouth_points[4]) # Distancia horizontal
    return (A + B) / (2.0 * C) if C > 0 else 0

def overlay_transparent(background_img, overlay_img, x, y, scale=1.0):
    """Superpone una imagen con transparencia sobre otra en la posición (x, y) centrada."""
    if overlay_img is None or scale <= 0 or overlay_img.shape[2] < 4:
        return background_img

    # 1. Redimensionar la imagen superpuesta y separar canales
    overlay_h, overlay_w = overlay_img.shape[:2]
    new_w = max(1, int(overlay_w * scale))
    new_h = max(1, int(overlay_h * scale))
    
    overlay = cv2.resize(overlay_img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    
    alpha = overlay[:, :, 3] / 255.0  # Máscara alfa (0.0 a 1.0)
    rgb = overlay[:, :, :3]           # Canales RGB

    # 2. Calcular límites del destino
    h_bg, w_bg = background_img.shape[:2]
    
    # Calcular el punto top-left (x1, y1)
    y1 = int(y - new_h / 2)
    x1 = int(x - new_w / 2)

    # Calcular el punto bottom-right (x2, y2) basándose en el tamaño
    y2 = y1 + new_h 
    x2 = x1 + new_w 

    # 3. Calcular límites de la *fuente* (en la imagen superpuesta)
    oy1, oy2 = 0, new_h
    ox1, ox2 = 0, new_w

    # 4. Recortar (Clip) contra los bordes de la imagen de fondo
    # Recortar arriba
    if y1 < 0:
        oy1 = -y1  
        y1 = 0     
    # Recortar abajo
    if y2 > h_bg:
        oy2 = new_h - (y2 - h_bg) 
        y2 = h_bg                 
    # Recortar izquierda
    if x1 < 0:
        ox1 = -x1
        x1 = 0
    # Recortar derecha
    if x2 > w_bg:
        ox2 = new_w - (x2 - w_bg)
        x2 = w_bg

    # 5. Si no hay área de superposición, salir
    if y2 <= y1 or x2 <= x1 or oy2 <= oy1 or ox2 <= ox1:
        return background_img

    # 6. Seleccionar Regiones de Interés (ROI)
    roi_bg = background_img[y1:y2, x1:x2]
    
    alpha_mask = alpha[oy1:oy2, ox1:ox2, np.newaxis] 
    rgb_overlay = rgb[oy1:oy2, ox1:ox2]

    # 7. Combinar imágenes
    roi_bg[:] = (1.0 - alpha_mask) * roi_bg + alpha_mask * rgb_overlay
    
    return background_img

def apply_magical_background(frame, sparkles_img, intensity=0.5):
    """Aplica un fondo rosa translúcido y destellos aleatorios."""
    h, w = frame.shape[:2]
    overlay = frame.copy()

    # Fondo rosa traslúcido
    pink_overlay = np.full((h, w, 3), (203, 195, 227), dtype=np.uint8) 
    cv2.addWeighted(pink_overlay, intensity, overlay, 1 - intensity, 0, overlay)

    # Añadir destellos aleatorios
    if sparkles_img is not None:
        num_sparkles = random.randint(15, 35)
        for _ in range(num_sparkles):
            scale = random.uniform(0.3, 1.3)
            sparkle_w = max(10, int(sparkles_img.shape[1] * scale))
            sparkle_h = max(10, int(sparkles_img.shape[0] * scale))

            # Redimensionar el destello 
            sparkle = cv2.resize(sparkles_img, (sparkle_w, sparkle_h))

            # Posición aleatoria (top-left)
            x_tl = random.randint(0, w - sparkle_w)
            y_tl = random.randint(0, h - sparkle_h)
            
            # Convertir a centro para la función overlay
            x_center = x_tl + sparkle_w // 2
            y_center = y_tl + sparkle_h // 2

            overlay = overlay_transparent(overlay, sparkle, x_center, y_center, scale=1.0) # scale=1.0 porque ya está redimensionado

    return overlay

def draw_unicorn_horn(frame, eyes_center, eye_distance, horn_img):
    """Dibuja el cuerno de unicornio sobre la frente."""
    if horn_img is None:
        return frame

    horn_x = eyes_center[0]
    forehead_height = eye_distance * 1.8
    horn_y = eyes_center[1] - int(forehead_height)

    target_width = eye_distance * 1.6
    base_scale = target_width / horn_img.shape[1]
    horn_scale = max(base_scale, 0.3)

    return overlay_transparent(frame, horn_img, horn_x, horn_y, horn_scale)

def draw_rainbow(frame, shape_points, rainbow_img):
    """Dibuja el arcoíris saliendo de la boca."""
    if rainbow_img is None:
        return frame
        
    mouth_top = shape_points[51]
    mouth_bottom = shape_points[57]
    
    mouth_center_x = (shape_points[48][0] + shape_points[54][0]) // 2 # Centro horizontal de la comisura
    mouth_center_y = (mouth_top[1] + mouth_bottom[1]) // 2

    mouth_width = np.linalg.norm(shape_points[48] - shape_points[54]) # Ancho real
    
    # Ajustes para posicionar el arcoíris
    offset_left = int(mouth_width * 0.25)
    offset_down = int(mouth_width * 1.9)

    rainbow_x = mouth_center_x - offset_left
    rainbow_y = mouth_center_y + offset_down

    target_width = mouth_width * 5.0
    base_scale = target_width / rainbow_img.shape[1]
    rainbow_scale = max(base_scale, 0.5)

    # Asegurar que el arcoíris no se salga por abajo
    overlay_h = int(rainbow_img.shape[0] * rainbow_scale)
    h_frame = frame.shape[0]
    rainbow_y = np.clip(rainbow_y, overlay_h // 2 + 20, h_frame - overlay_h // 2 - 20)

    return overlay_transparent(frame, rainbow_img, rainbow_x, rainbow_y, rainbow_scale)


# Configuración 
MOUTH_AR_THRESH = 0.50
detector = FaceDetector()

# Cargar imágenes
try:
    unicorn_img = cv2.imread("assets/unicorn_horn.png", cv2.IMREAD_UNCHANGED)
    rainbow_img = cv2.imread("assets/rainbow.png", cv2.IMREAD_UNCHANGED)
    sparkles_img = cv2.imread("assets/sparkles.png", cv2.IMREAD_UNCHANGED)
    
    if unicorn_img is None or rainbow_img is None or sparkles_img is None:
        raise FileNotFoundError("Una o más imágenes no se encontraron.")
except Exception as e:
    print(f"Error cargando imágenes: {e}")
    print("Asegúrate de tener en la carpeta:")
    print("   - unicorn_horn.png")
    print("   - rainbow.png")
    print("   - sparkles.png (con fondo transparente)")
    unicorn_img, rainbow_img, sparkles_img = None, None, None

# Iniciar cámara
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Error: No se pudo abrir la cámara.")
    exit()

print("Filtro Unicornio - Presiona 'q' para salir")
print("Abre la boca para activar: CUERNO + ARCOÍRIS + FONDO MÁGICO")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Error: No se pudo leer el frame.")
        break

    frame = cv2.flip(frame, 1)
    output = frame.copy()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    values = detector.SingleFaceEyesDetection(gray, facedet='DLIB', eyesdet='DLIB68')
    
    if values is not None:
        face_rect, eye_coords, shape = values
        
        if len(shape) == 68:
            # Índices faciales
            (l_start, l_end) = face_utils.FACIAL_LANDMARKS_IDXS["left_eye"]
            (r_start, r_end) = face_utils.FACIAL_LANDMARKS_IDXS["right_eye"]
            (m_start, m_end) = face_utils.FACIAL_LANDMARKS_IDXS["inner_mouth"]

            # Calcular geometría 
            left_eye_center = np.mean(shape[l_start:l_end], axis=0).astype(int)
            right_eye_center = np.mean(shape[r_start:r_end], axis=0).astype(int)
            eyes_center = (
                (left_eye_center[0] + right_eye_center[0]) // 2,
                (left_eye_center[1] + right_eye_center[1]) // 2
            )
            eye_distance = max(np.linalg.norm(left_eye_center - right_eye_center), 20)

            mouth_points = shape[m_start:m_end]
            mar = get_mouth_aspect_ratio(mouth_points)

            # Mostrar MAR 
            cv2.putText(output, f"MAR: {mar:.2f}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            # Comprobar si la boca está abierta 
            if mar > MOUTH_AR_THRESH:
                # Aplicar efectos
                output = apply_magical_background(output, sparkles_img, intensity=0.55)
                output = draw_unicorn_horn(output, eyes_center, eye_distance, unicorn_img)
                output = draw_rainbow(output, shape, rainbow_img)

            else:
                # Boca cerrada
                cv2.putText(output, "Abre la boca para magia!", (10, 60),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 100, 100), 2)
        else:
            cv2.putText(output, "Cara no detectada (68 puntos)", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    else:
        cv2.putText(output, "No se detectó rostro", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    # Mostrar resultado
    cv2.imshow("Unicornio", output)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# --- Limpieza ---
cap.release()
cv2.destroyAllWindows()
print("¡Filtro cerrado! ¡Hasta la próxima magia!")